# Fonts !

## Read raw data and use kmeans to standardize the number of points

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances_argmin_min

nonxy_cols = ["font", "char"]

def points_to_xy(points):
  xs = points[:, 0]
  ys = points[:, 1]
  return np.stack((xs, ys), axis=1).reshape(-1)

with open("./json/fonts_p5_raw.json", "r") as ifp:
  fonts_raw = json.load(ifp)

In [ ]:
min_points_len = min([len(l["points"]) for l in fonts_raw] + [480])
print(min_points_len)

letter_info = np.array([[l[k] for k in nonxy_cols] for l in fonts_raw])
letter_info_df = pd.DataFrame(letter_info, columns=nonxy_cols)

char_list = np.sort(letter_info_df["char"].unique()).tolist()

In [ ]:
letter_contours = []

for idx,letter in enumerate(fonts_raw):
  points = np.array(letter["points"])

  kmeans = KMeans(n_clusters=min_points_len, random_state=1010).fit(points)

  contour_idxs, _ = pairwise_distances_argmin_min(kmeans.cluster_centers_, points)

  if len(contour_idxs) != len(list(set(contour_idxs))):
    print("have duplicate points in contour")

  contour_points = points[contour_idxs]

  letter_contours.append(points_to_xy(contour_points))
  if idx % 100 == 0:
    print(idx, "/", len(fonts_raw))

letter_contours_np = np.array(letter_contours)
letter_contours_np.shape

In [ ]:
contour_cols = np.array([(f"x{i}", f"y{i}") for i in range(letter_contours_np.shape[1]//2)]).reshape(-1).tolist()

letter_contours_df = pd.DataFrame(letter_contours_np, columns=contour_cols).round(6)

fonts_df = pd.concat([letter_info_df, letter_contours_df], axis=1)

fonts_df.to_csv(f"./csv/fonts_{min_points_len}_raw.csv", index=False)

In [ ]:
idx = char_list.index("T") + 62 * 0
points = fonts_df.loc[idx:idx].drop(columns=nonxy_cols).values.reshape(-1,2)
avg_points = fonts_df[fonts_df["char"] == "T"].drop(columns=nonxy_cols).mean().values.reshape(-1, 2)
xs = points[:,0]
ys = -points[:,1]

plt.axis("equal")
plt.plot(xs, ys, marker="o", markersize=4, linestyle="", alpha=0.3)
plt.plot(xs.mean(), ys.mean(), marker="x", markersize=8, color="red")
plt.plot((xs.max() + xs.min())/2, (ys.max() + ys.min())/2, marker="x", markersize=8, color="green")
plt.show()

plt.axis("equal")
plt.plot(avg_points[:,0], -avg_points[:,1], marker="o", markersize=4, linestyle="", alpha=0.3)
plt.show()

## Order points

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.preprocessing import MaxAbsScaler, StandardScaler

def polar_dist(xy):
  x,y=xy
  r = (x**2 + y**2) ** 0.5
  a = (np.arctan2(y, x) + np.pi)
  return 100*a + r

def center_points(ps):
  pmin = ps.min(axis=0)
  pmax = ps.max(axis=0)
  center = (pmin + pmax) / 2
  return ps - center

def average_points(ps):
  average = ps.mean(axis=0)
  return ps - average

scaler = StandardScaler()

fonts_df = pd.read_csv("./csv/fonts_512_raw.csv")

xy_cols = [c for c in fonts_df.columns if c.startswith(("x", "y"))]
nonxy_cols = [c for c in fonts_df.columns if not c.startswith(("x", "y"))]
char_list = np.sort(fonts_df["char"].unique()).tolist()
nrows = len(fonts_df)

In [ ]:
font_points_np = fonts_df.drop(columns=nonxy_cols).values.reshape(nrows, -1, 2)
font_center_np = np.apply_along_axis(average_points, axis=1, arr=font_points_np)
font_polar_np = np.apply_along_axis(polar_dist, axis=2, arr=font_center_np)
font_polar_order = np.argsort(font_polar_np, axis=1)

In [ ]:
font_ordered_np = np.take_along_axis(font_center_np, font_polar_order[:, :, np.newaxis], axis=1)
fonts_scaled_np = scaler.fit_transform(font_ordered_np.reshape(nrows, -1).T).T
fonts_scaled_df = pd.DataFrame(fonts_scaled_np, columns=xy_cols)
fonts_scaled_df = pd.concat((fonts_df[nonxy_cols], fonts_scaled_df), axis=1)

In [ ]:
ml = "T"
idx = char_list.index(ml) + 62 * 1
points = fonts_scaled_df.loc[idx:idx].drop(columns=nonxy_cols).values.reshape(-1,2)
avg_points = fonts_scaled_df[fonts_scaled_df["char"] == ml].drop(columns=nonxy_cols).mean().values.reshape(-1, 2)
avg_all = fonts_scaled_df.drop(columns=nonxy_cols).mean(axis=0).values.reshape(-1, 2)
xs = points[:,0]
ys = -points[:,1]

plt.axis("equal")
plt.plot(xs, ys, marker="o", markersize=4, linestyle="-", alpha=0.3)
plt.plot(xs.mean(), ys.mean(), marker="x", markersize=8, color="red")
plt.plot((xs.max() + xs.min())/2, (ys.max() + ys.min())/2, marker="x", markersize=8, color="green")
plt.show()

plt.axis("equal")
plt.plot(avg_points[:,0], -avg_points[:,1], marker="o", markersize=4, linestyle="", alpha=0.3)
plt.show()

plt.axis("equal")
plt.plot(avg_all[:,0], -avg_all[:,1], marker="o", markersize=4, linestyle="", alpha=0.3)
plt.show()

In [ ]:
fonts_scaled_df.to_csv(f"./csv/fonts_{len(xy_cols)//2}_ordered.csv", index=False)

## PCA

### PCA and components per character

In [ ]:
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA

fonts_ordered_df = pd.read_csv("./csv/fonts_480_ordered.csv")

xy_cols = [c for c in fonts_ordered_df.columns if c.startswith(("x", "y"))]
nonxy_cols = [c for c in fonts_ordered_df.columns if not c.startswith(("x", "y"))]
char_list = np.sort(fonts_ordered_df["char"].unique()).tolist()
nrows = len(fonts_ordered_df)
nfonts = nrows//len(char_list)
npoints = len(xy_cols)//2

In [ ]:
fonts_pca_df = pd.DataFrame()
fonts_components_df = pd.DataFrame()

for c in char_list:
  mpca = PCA(n_components=min(nfonts, npoints)).set_output(transform="pandas")
  char_df = fonts_ordered_df[(fonts_ordered_df["char"] == c)]
  char_pca_df = mpca.fit_transform(char_df.drop(columns=nonxy_cols))
  char_pca_df = pd.concat((char_df[nonxy_cols], char_pca_df), axis=1)
  fonts_pca_df = pd.concat((fonts_pca_df, char_pca_df), axis=0)

  char_components_df = pd.DataFrame(np.concatenate(([mpca.mean_], mpca.components_)), columns=xy_cols)
  char_components_df.insert(0, "char", c)
  fonts_components_df = pd.concat((fonts_components_df, char_components_df), axis=0)

In [ ]:
fonts_pca_df = fonts_pca_df.sort_index()
fonts_pca_df = fonts_pca_df.round({c:6 for c in fonts_pca_df.columns if c not in nonxy_cols})
fonts_components_df = fonts_components_df.round({c:6 for c in fonts_ordered_df.columns if c not in nonxy_cols}).reset_index(drop=True)

In [ ]:
fonts_pca_df.to_csv(f"./csv/fonts_{npoints}_pca_pcs.csv", index=False)
fonts_components_df.to_csv(f"./csv/fonts_{npoints}_pca_components.csv", index=False)

### Test

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

fonts_pca_df = pd.read_csv("./csv/fonts_480_pca_pcs.csv")
fonts_components_df = pd.read_csv("./csv/fonts_480_pca_components.csv")

xy_cols = [c for c in fonts_components_df.columns if c.startswith(("x", "y"))]
nonxy_cols = [c for c in fonts_components_df.columns if not c.startswith(("x", "y"))]
char_list = np.sort(fonts_components_df["char"].unique()).tolist()

In [ ]:
char_components = {}

for c in char_list:
  char_components[c] = fonts_components_df[fonts_components_df["char"] == c].drop(columns=nonxy_cols).values

In [ ]:
avgPCs = 0.0 * np.ones((1,41))
ml = "A"
points = (avgPCs @ char_components[ml][1:] + char_components[ml][0]).reshape(-1, 2)
xs = points[:,0]
ys = -points[:,1]

plt.axis("equal")
plt.plot(xs, ys, marker="o", markersize=4, linestyle="", alpha=0.5)
plt.show()

### PCA everything

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import euclidean_distances

fonts_ordered_df = pd.read_csv("./csv/fonts_480_ordered.csv")

xy_cols = [c for c in fonts_ordered_df.columns if c.startswith(("x", "y"))]
nonxy_cols = [c for c in fonts_ordered_df.columns if not c.startswith(("x", "y"))]
char_list = np.sort(fonts_ordered_df["char"].unique()).tolist()
nrows = len(fonts_ordered_df)

In [ ]:
mpca = PCA(n_components=41)

fonts_pca_np = mpca.fit_transform(fonts_ordered_df.drop(columns=nonxy_cols))
print(sum(mpca.explained_variance_ratio_), mpca.n_components_)

pca_dists = euclidean_distances(fonts_pca_np, fonts_pca_np)
pca_dists_sorted = pca_dists.argsort(axis=1)

fonts_ipca_np = mpca.inverse_transform(fonts_pca_np).reshape(nrows, -1)
fonts_ipca_df = pd.DataFrame(fonts_ipca_np, columns=xy_cols)
fonts_ipca_df = pd.concat((fonts_ordered_df[nonxy_cols], fonts_ipca_df), axis=1)

In [ ]:
# points = mpca.components_[0].reshape(-1,2)
points = fonts_ordered_df.drop(columns=nonxy_cols).mean(axis=0).values.reshape(-1,2)
xs = points[:,0]
ys = -points[:,1]

plt.axis("equal")
plt.plot(xs, ys, marker="o", markersize=4, linestyle="", alpha=0.5)
plt.show()

In [ ]:
ml = "A"
idx = char_list.index(ml) + 62 * 1
points = fonts_ipca_df.loc[idx:idx].drop(columns=nonxy_cols).values.reshape(-1,2)
xs = points[:,0]
ys = -points[:,1]

plt.axis("equal")
plt.plot(xs, ys, marker="o", markersize=4, linestyle="", alpha=0.5)
plt.show()

In [ ]:
ml = "A"
idx = char_list.index(ml) + 62 * 1

for idx in pca_dists_sorted[idx, :8]:
  points = fonts_ordered_df.loc[idx:idx].drop(columns=nonxy_cols).values.reshape(-1,2)
  xs = points[:,0]
  ys = -points[:,1]

  plt.axis("equal")
  plt.plot(xs, ys, marker="o", markersize=4, linestyle="")
  plt.show()